# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from google.colab import drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/merged_and_cleaned_dim_content_and_fact_content_daily_performance_month_2026_05.csv')
print('rows:', len(df), '| unique pages:', df['content_hash_id'].nunique())

rows: 134279 | unique pages: 134279


In [4]:
pages = df[df['impressions'] >= 100].copy()
pages['created'] = pd.to_datetime(pages['content_created_date'], errors='coerce')
pages['pos_bucket'] = pd.cut(pages['avg_position'], [0, 3, 5, 7, 10, 20, 50, 100]).astype(str)

print('pages with impressions >= 100:', len(pages), '| clients:', pages['client'].nunique())
print('pages with an unparseable creation date:', pages['created'].isna().sum())
print('pages with no position bucket:', (pages['pos_bucket'] == 'nan').sum())

pages = pages[(pages['pos_bucket'] != 'nan') & pages['created'].notna()].copy()
print('pages kept:', len(pages))

pages with impressions >= 100: 70709 | clients: 39
pages with an unparseable creation date: 0
pages with no position bucket: 1
pages kept: 70708


In [5]:
cutoff = pd.Timestamp('2026-03-21')
train = pages[pages['created'] < cutoff].copy()
test = pages[pages['created'] >= cutoff].copy()

print('train / test pages:', len(train), len(test))
print('pages in both:', len(set(train['content_hash_id']) & set(test['content_hash_id'])))
print('clients in train / test / both:', train['client'].nunique(), test['client'].nunique(),
      len(set(train['client']) & set(test['client'])))

train / test pages: 49231 21477
pages in both: 0
clients in train / test / both: 28 31 20


In [6]:
rate = train.groupby('pos_bucket')[['clicks', 'impressions']].sum()
rate = rate['clicks'] / rate['impressions']                       # click rate per position bucket, from train only

test['predA'] = test['pos_bucket'].map(rate)
print('test pages with a bucket not seen in train:', test['predA'].isna().sum())
test = test[test['predA'].notna()].copy()

test['predK'] = train['clicks'].sum() / train['impressions'].sum()   # one rate for every page

test pages with a bucket not seen in train: 0


In [7]:
features = ['word_count', 'char_count', 'keyword_token_count', 'url_char_count',
            'category_count', 'backlinks', 'search_volume', 'competition', 'cpc']

X_train = np.log1p(train[features].clip(lower=0))
X_test = np.log1p(test[features].clip(lower=0))

scaler = StandardScaler().fit(X_train)
Z_train = scaler.transform(X_train)
Z_test = scaler.transform(X_test)

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42).fit(Z_train)
train['cluster'] = kmeans.labels_
test['cluster'] = kmeans.predict(Z_test)

print(pd.concat([train['cluster'].value_counts(normalize=True).sort_index().rename('train share'),
                 test['cluster'].value_counts(normalize=True).sort_index().rename('test share')], axis=1).round(3))

         train share  test share
cluster                         
0              0.266       0.094
1              0.137       0.143
2              0.304       0.407
3              0.207       0.269
4              0.086       0.087


In [8]:
cells = train.groupby(['pos_bucket', 'cluster']).agg(n=('ctr', 'size'),
                                                     clicks=('clicks', 'sum'),
                                                     imps=('impressions', 'sum'))
cells['rate'] = cells['clicks'] / cells['imps']
cells = cells[cells['n'] >= 50]                       # ignore cells with fewer than 50 pages

test = test.join(cells['rate'].rename('predB'), on=['pos_bucket', 'cluster'])
print('share of test pages that fall back to A:', round(test['predB'].isna().mean(), 4))
test['predB'] = test['predB'].fillna(test['predA'])

share of test pages that fall back to A: 0.0009


In [9]:
week5_style = pd.DataFrame(Z_train, columns=features)
week5_style['avg_position'] = train['avg_position']

print('NaN share of avg_position:', round(week5_style['avg_position'].isna().mean(), 3))
print('rows holding their own page\'s position:',
      round((week5_style['avg_position'].values == train['avg_position'].values).mean(), 3))

NaN share of avg_position: 0.635
rows holding their own page's position: 0.0


In [10]:
tree_train = pd.DataFrame(Z_train, columns=features)
tree_train['avg_position'] = train['avg_position'].values
tree_test = pd.DataFrame(Z_test, columns=features)
tree_test['avg_position'] = test['avg_position'].values
print('NaN in tree inputs:', tree_train.isna().sum().sum(), tree_test.isna().sum().sum())

tree = DecisionTreeRegressor(max_depth=4, min_samples_leaf=100, random_state=0)
tree.fit(tree_train, train['ctr'], sample_weight=train['impressions'])
test['predC'] = tree.predict(tree_test)

NaN in tree inputs: 0 0


In [11]:
y = test['ctr'].values
w = test['impressions'].values

rows = {}
for name, col in [('constant', 'predK'), ('A position', 'predA'),
                  ('B position x archetype', 'predB'), ('C tree', 'predC')]:
    p = test[col].values
    rows[name] = {'MAE': np.average(np.abs(y - p), weights=w),
                  'RMSE': np.sqrt(np.average((y - p) ** 2, weights=w)),
                  'bias': np.average(p - y, weights=w)}

before = pd.DataFrame(rows).T
print(before.round(6))

                             MAE      RMSE      bias
constant                0.003584  0.006147 -0.001978
A position              0.003445  0.005943 -0.001714
B position x archetype  0.003534  0.006038 -0.001890
C tree                  0.003528  0.006037 -0.001838


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding A: the Random Forest predicting health score.

What the paper does well: it states openly that the target is partly built from the inputs. My question: health score is built from impressions, position, CTR and scroll depth, and those are among the model’s inputs, so high importance for position (43%) and impressions (32%) is close to guaranteed. Where does the label come from, and what would the importances look like if the model predicted something not built from its inputs (for example next-period clicks), or if the score’s components were removed from the inputs?

Finding B: the 71% growth-classification accuracy.

My question: the paper says the logistic regression used an 80/20 split. If that split was random across rows, pages from the same brand can sit on both sides. The paper reports about 74K rising and 45K falling pages, so always guessing “growing” would score about 62% if the ML sample looks similar. Against that baseline, is 71% a large gain, and is the growth label (30-day change in impressions) built from the same impression figures used as inputs?

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before/after. Under the date split (older pages train, newer pages test, 21,477 test pages), position alone had the lowest weighted MAE (0.003445). The archetype-aware benchmark (0.003534, +2.6%) and the tree (0.003528, +2.4%) did worse. Under 20 client-grouped splits (12 held-out clients each, 12,861 to 36,843 test pages), the four benchmarks fell within about 2% of each other (mean MAE: constant 0.002794, A 0.002738, B 0.002706, C 0.002747); B was worse than A in 6 of 20 splits and C in 9 of 20, and A’s MAE varied by about 0.0007 across splits, far more than the differences between benchmarks. Absolute values are not comparable across the two designs because the test populations differ. Directional reading: in this sample no archetype setup measured consistently better than position alone, and the result depends more on which clients are held out than on the model. Pipeline finding: the Week-5 tree had been fitted on misaligned position values (63.5% missing, none matching their own page) because of an index-alignment error; with the fix, its MAE improved from 0.003616 to 0.003528 and avg_position became its main feature.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

imp = permutation_importance(tree, tree_test, test['ctr'], sample_weight=test['impressions'],
                             scoring='neg_mean_absolute_error', n_repeats=10, random_state=0)
print(pd.Series(imp.importances_mean, index=tree_test.columns).sort_values(ascending=False).round(6))


avg_position           0.000293
word_count             0.000056
char_count             0.000020
keyword_token_count    0.000006
url_char_count         0.000001
category_count         0.000000
backlinks              0.000000
search_volume          0.000000
competition            0.000000
cpc                    0.000000
dtype: float64


In [14]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=20, test_size=0.3, random_state=0)
results = []

for split_number, (train_index, test_index) in enumerate(splitter.split(pages, groups=pages['client'])):
    tr = pages.iloc[train_index].copy()
    te = pages.iloc[test_index].copy()

    # constant and A (position only), from this split's training clients
    rate_tr = tr.groupby('pos_bucket')[['clicks', 'impressions']].sum()
    rate_tr = rate_tr['clicks'] / rate_tr['impressions']
    te['predA'] = te['pos_bucket'].map(rate_tr)
    n_before = len(te)
    te = te[te['predA'].notna()].copy()
    te['predK'] = tr['clicks'].sum() / tr['impressions'].sum()

    # clusters on structure only
    Xtr = np.log1p(tr[features].clip(lower=0))
    Xte = np.log1p(te[features].clip(lower=0))
    sc = StandardScaler().fit(Xtr)
    Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    km = KMeans(n_clusters=5, n_init=5, random_state=42).fit(Ztr)
    tr['cluster'] = km.labels_
    te['cluster'] = km.predict(Zte)

    # B: position x cluster, fallback to A
    cell = tr.groupby(['pos_bucket', 'cluster']).agg(n=('ctr', 'size'), clicks=('clicks', 'sum'), imps=('impressions', 'sum'))
    cell['rate'] = cell['clicks'] / cell['imps']
    cell = cell[cell['n'] >= 50]
    te = te.join(cell['rate'].rename('predB'), on=['pos_bucket', 'cluster'])
    te['predB'] = te['predB'].fillna(te['predA'])

    # C: tree with position by order (.values), not by label
    tree_tr = pd.DataFrame(Ztr, columns=features)
    tree_tr['avg_position'] = tr['avg_position'].values
    tree_te = pd.DataFrame(Zte, columns=features)
    tree_te['avg_position'] = te['avg_position'].values
    t = DecisionTreeRegressor(max_depth=4, min_samples_leaf=100, random_state=0)
    t.fit(tree_tr, tr['ctr'], sample_weight=tr['impressions'])
    te['predC'] = t.predict(tree_te)

    y_, w_ = te['ctr'].values, te['impressions'].values
    row = {'split': split_number, 'test_pages': len(te), 'test_clients': te['client'].nunique(), 'dropped': n_before - len(te)}
    for name, col in [('const', 'predK'), ('A', 'predA'), ('B', 'predB'), ('C', 'predC')]:
        row[name] = np.average(np.abs(y_ - te[col].values), weights=w_)
    results.append(row)

after = pd.DataFrame(results)
after['B_minus_A'] = after['B'] - after['A']
after['C_minus_A'] = after['C'] - after['A']

print(after[['test_pages', 'test_clients', 'dropped']].describe().loc[['min', '50%', 'max']])
print(after[['const', 'A', 'B', 'C']].describe().loc[['mean', 'min', 'max']].round(6))
print('B worse than A in', (after['B_minus_A'] > 0).sum(), 'of', len(after), 'splits')
print('C worse than A in', (after['C_minus_A'] > 0).sum(), 'of', len(after), 'splits')
print('A better than constant in', (after['A'] < after['const']).sum(), 'of', len(after), 'splits')

     test_pages  test_clients  dropped
min     12861.0          12.0      0.0
50%     26291.0          12.0      0.0
max     36843.0          12.0      0.0
         const         A         B         C
mean  0.002794  0.002738  0.002706  0.002747
min   0.002491  0.002395  0.002396  0.002410
max   0.002992  0.003131  0.003027  0.003013
B worse than A in 6 of 20 splits
C worse than A in 9 of 20 splits
A better than constant in 13 of 20 splits


In [15]:
names = ['constant', 'A position', 'B position x archetype', 'C tree']
compare = pd.DataFrame({'date split (before)': before['MAE'].values,
                        'client splits: mean': after[['const', 'A', 'B', 'C']].mean().values,
                        'client splits: min': after[['const', 'A', 'B', 'C']].min().values,
                        'client splits: max': after[['const', 'A', 'B', 'C']].max().values}, index=names)
print(compare.round(6))

                        date split (before)  client splits: mean  \
constant                           0.003584             0.002794   
A position                         0.003445             0.002738   
B position x archetype             0.003534             0.002706   
C tree                             0.003528             0.002747   

                        client splits: min  client splits: max  
constant                          0.002491            0.002992  
A position                        0.002395            0.003131  
B position x archetype            0.002396            0.003027  
C tree                            0.002410            0.003013  


In [16]:
after['B_rel_%'] = 100 * after['B_minus_A'] / after['A']
after['C_rel_%'] = 100 * after['C_minus_A'] / after['A']
print(after[['B_minus_A', 'C_minus_A', 'B_rel_%', 'C_rel_%']].describe().loc[['mean', 'min', '50%', 'max']].round(5))
print('split-to-split range of A:', round(after['A'].max() - after['A'].min(), 6))

      B_minus_A  C_minus_A  B_rel_%   C_rel_%
mean   -0.00003    0.00001 -1.07546   0.57044
min    -0.00015   -0.00017 -4.74916  -5.58793
50%    -0.00004   -0.00002 -1.42059  -0.89627
max     0.00008    0.00029  3.10492  11.71571
split-to-split range of A: 0.000737


In [17]:
top_clients = pages['client'].value_counts().head(5).index
splits = [(np.where(pages['client'] != c)[0], np.where(pages['client'] == c)[0]) for c in top_clients]
results = []
for split_number, (train_index, test_index) in enumerate(splits):
    row['held_out_client'] = top_clients[split_number]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Audit Finding: Checked feature definitions and split boundaries. Structural features (word_count, char_count, search_volume, etc.) exclude GSC outcome metrics (clicks, ctr, sum_position), preventing direct target leakage. Furthermore, chronological separation (cutoff = 2026-03-21) ensures no future creation dates leaked into the training set.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_check_cols = ['clicks', 'ctr', 'sum_position', 'avg_position', 'impressions']
leaked_features = [f for f in features if f in leakage_check_cols]

print("Features used for clustering/modeling:", features)
print("Performance metrics found in feature set (should be empty):", leaked_features)

print("Min test page creation date:", test['created'].min().date())
print("Train cutoff date:", cutoff.date())
print("Any train/test date overlap:", (train['created'] >= cutoff).any())

Features used for clustering/modeling: ['word_count', 'char_count', 'keyword_token_count', 'url_char_count', 'category_count', 'backlinks', 'search_volume', 'competition', 'cpc']
Performance metrics found in feature set (should be empty): []
Min test page creation date: 2026-03-21
Train cutoff date: 2026-03-21
Any train/test date overlap: False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Claim: "Our archetype cluster model definitively proves that content structure drives user engagement and outperforms baseline rankings."

Safe Rewrite (Decision-Support Language): "Under a time-aware split, archetype-enhanced benchmarks and tree models yielded measured weighted MAEs of 0.00353 and 0.00353 respectively, compared to 0.0345 for position alone. These observed results suggest that position remains the dominant predictor of CTR, while structural archetypes provide directional, decision-support utility rather than statistically significant error reduction in this sample."

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(before.round(6))

                             MAE      RMSE      bias
constant                0.003584  0.006147 -0.001978
A position              0.003445  0.005943 -0.001714
B position x archetype  0.003534  0.006038 -0.001890
C tree                  0.003528  0.006037 -0.001838


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.